# Operational Context Graph — Cross-Demo Comparison

This notebook compares the AI4I and SECOM demos side-by-side across four dimensions:

| Dimension | What it measures |
|---|---|
| **Graph statistics** | How many nodes, relationships, entity types each demo produced |
| **Risk detection** | How many items each risk function flagged and at what levels |
| **Decision latency** | How quickly the system can score a single entity vs. a batch |
| **Traceability** | Whether every governed action is fully auditable |

**Key question this notebook answers:**  
Did adding functions, actions, and governance on top of the knowledge graph actually give us something a plain dashboard couldn't?

---

### Before you run this

```bash
# Infrastructure must be running
docker compose -f infra/docker-compose.yml up -d

# Data must be loaded
uv run python -m demos.ai4i.load
uv run python -m demos.secom.load
```

In [ ]:
import time
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.ontology.graph import GraphSession
from core.governance.db import SessionLocal
from demos.ai4i.track_b.functions import PredictFailureRisk
from demos.secom.track_b.functions import PredictYieldRisk
from sqlalchemy import text

graph = GraphSession.from_env()
graph.verify_connectivity()
print("Neo4j connected.")

ai4i_fn = PredictFailureRisk()
secom_fn = PredictYieldRisk()
print("Functions loaded.")

---

## 1. Graph Statistics

The most basic comparison: what did each demo actually put into Neo4j?

In [ ]:
def count_nodes(label: str) -> int:
    rows = graph.run(f"MATCH (n:{label}) RETURN count(n) AS c")
    return rows[0]["c"] if rows else 0

def count_rels(rel_type: str) -> int:
    rows = graph.run(f"MATCH ()-[r:{rel_type}]->() RETURN count(r) AS c")
    return rows[0]["c"] if rows else 0

stats = pd.DataFrame([
    # AI4I
    {"demo": "AI4I", "entity": "Machine",      "count": count_nodes("Machine")},
    {"demo": "AI4I", "entity": "ToolRun",      "count": count_nodes("ToolRun")},
    {"demo": "AI4I", "entity": "FailureMode",  "count": count_nodes("FailureMode")},
    {"demo": "AI4I", "entity": "HAS_RUN",      "count": count_rels("HAS_RUN")},
    {"demo": "AI4I", "entity": "HAS_FAILURE",  "count": count_rels("HAS_FAILURE")},
    # SECOM
    {"demo": "SECOM", "entity": "Lot",             "count": count_nodes("Lot")},
    {"demo": "SECOM", "entity": "YieldOutcome",    "count": count_nodes("YieldOutcome")},
    {"demo": "SECOM", "entity": "SPCAlarm",        "count": count_nodes("SPCAlarm")},
    {"demo": "SECOM", "entity": "HAS_OUTCOME",     "count": count_rels("HAS_OUTCOME")},
    {"demo": "SECOM", "entity": "TRIGGERED_ALARM", "count": count_rels("TRIGGERED_ALARM")},
])

print(stats.to_string(index=False))

In [ ]:
# Summary totals
summary = pd.DataFrame([
    {"demo": "AI4I",  "total_nodes": 10008,  "total_rels": 10373,
     "entity_types": 3, "rel_types": 2,
     "primary_entities": "ToolRun", "primary_count": 10000},
    {"demo": "SECOM", "total_nodes": 2009,   "total_rels": 7682,
     "entity_types": 3, "rel_types": 2,
     "primary_entities": "Lot",     "primary_count": 1567},
])
summary

**Observation:** AI4I has more primary entities (10K ToolRuns) but a sparser graph (373 failure links = 3.7% of runs had a recorded failure). SECOM has fewer primary entities (1,567 Lots) but a denser set of derived relationships (6,115 SPC alarm links = average ~3.9 alarms per lot). The graph structure itself is informative — dense alarm clusters are a signal even before scoring.

---

## 2. Dataset Characteristics

The two datasets differ in more than size — their failure regimes are fundamentally different.

In [ ]:
characteristics = pd.DataFrame([
    {"Property": "Domain",             "AI4I": "Generic milling machine",        "SECOM": "Semiconductor wafer fab"},
    {"Property": "Primary entities",   "AI4I": "10,000 ToolRuns",                "SECOM": "1,567 Lots"},
    {"Property": "Features per entity","AI4I": "5 sensor readings",              "SECOM": "590 sensor readings"},
    {"Property": "Failure rate",        "AI4I": "3.39% (339/10,000)",            "SECOM": "6.64% (104/1,567)"},
    {"Property": "Timestamps",          "AI4I": "Synthetic (6-min intervals)",   "SECOM": "Real (Jul–Oct 2008)"},
    {"Property": "Missing data",        "AI4I": "None",                          "SECOM": "4.5% overall, up to 100% per feature"},
    {"Property": "Failure types",       "AI4I": "5 named types (TWF/HDF/PWF/OSF/RNF)", "SECOM": "Binary pass/fail only"},
    {"Property": "Derived relationships","AI4I": "HAS_FAILURE (rule: flag present)", "SECOM": "TRIGGERED_ALARM (rule: |σ| > 3)"},
    {"Property": "Graph enrichment",   "AI4I": "Shared failure-type nodes reused across runs", "SECOM": "Shared alarm-feature nodes with control limits"},
])
characteristics

**Observation:** The entity mapping exercise is where the real learning happens. Both datasets map onto the same vocabulary (entity → property → link → derived property → action) but the translation looks very different:

- AI4I: failure modes are already explicit labels → natural graph nodes  
- SECOM: failure signal is implicit in sensor deviation patterns → must derive the "SPCAlarm" structure

In a real fab, you'd face this every time: some signals are explicit events, others must be computed.

---

## 3. Risk Detection Results

How many entities did each function flag, and at what levels?

In [ ]:
# Score AI4I ToolRuns
ai4i_runs = graph.run("""
    MATCH (r:ToolRun)
    RETURN r.air_temp_k AS air_temp_k,
           r.process_temp_k AS process_temp_k,
           r.rotational_speed_rpm AS rotational_speed_rpm,
           r.torque_nm AS torque_nm,
           r.tool_wear_min AS tool_wear_min,
           r.machine_id AS machine_id,
           r.machine_failure AS machine_failure
""")

ai4i_scores = [ai4i_fn.compute(r) for r in ai4i_runs]
ai4i_df = pd.DataFrame(ai4i_scores)
ai4i_df["actual_failure"] = [r["machine_failure"] for r in ai4i_runs]

ai4i_summary = ai4i_df["risk_level"].value_counts().rename_axis("risk_level").reset_index(name="count")
ai4i_summary["demo"] = "AI4I"
ai4i_summary["pct"] = (ai4i_summary["count"] / len(ai4i_df) * 100).round(1)
print("AI4I risk distribution:")
print(ai4i_summary.to_string(index=False))

In [ ]:
# Score SECOM Lots
secom_lots = graph.run("""
    MATCH (lot:Lot)
    OPTIONAL MATCH (lot)-[r:TRIGGERED_ALARM]->(:SPCAlarm)
    WITH lot, max(abs(r.sigma_deviation)) AS max_sigma
    RETURN lot.n_spc_alarms   AS n_spc_alarms,
           lot.sensor_na_rate AS sensor_na_rate,
           coalesce(max_sigma, 0.0) AS max_sigma_dev,
           lot.yield_pass     AS yield_pass
""")

secom_scores = [secom_fn.compute(r) for r in secom_lots]
secom_df = pd.DataFrame(secom_scores)
secom_df["actual_pass"] = [r["yield_pass"] for r in secom_lots]

secom_summary = secom_df["risk_level"].value_counts().rename_axis("risk_level").reset_index(name="count")
secom_summary["demo"] = "SECOM"
secom_summary["pct"] = (secom_summary["count"] / len(secom_df) * 100).round(1)
print("SECOM risk distribution:")
print(secom_summary.to_string(index=False))

In [ ]:
# Side-by-side chart
combined = pd.concat([ai4i_summary, secom_summary], ignore_index=True)
fig = px.bar(
    combined,
    x="demo", y="count", color="risk_level", barmode="group",
    color_discrete_map={"high": "#d62728", "medium": "#ff7f0e", "low": "#2ca02c"},
    text="pct",
    title="Risk level distribution: AI4I vs SECOM",
    labels={"count": "Entity count"},
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [ ]:
# Precision check: of high-risk entities, how many actually failed?
ai4i_high = ai4i_df[ai4i_df["risk_level"] == "high"]
ai4i_precision = ai4i_high["actual_failure"].mean() if len(ai4i_high) else 0
ai4i_recall = (
    ai4i_df[ai4i_df["actual_failure"] == True]["risk_level"]
    .isin(["high", "medium"]).mean()
)

secom_high = secom_df[secom_df["risk_level"] == "high"]
secom_precision = (secom_high["actual_pass"] == False).mean() if len(secom_high) else 0
secom_recall = (
    secom_df[secom_df["actual_pass"] == False]["risk_level"]
    .isin(["high", "medium"]).mean()
)

quality = pd.DataFrame([
    {"Demo": "AI4I",  "High-risk entities": len(ai4i_high),
     "Precision (high-risk actually failed)": f"{ai4i_precision:.1%}",
     "Recall (failures caught at high/medium)": f"{ai4i_recall:.1%}"},
    {"Demo": "SECOM", "High-risk entities": len(secom_high),
     "Precision (high-risk actually failed)": f"{secom_precision:.1%}",
     "Recall (failures caught at high/medium)": f"{secom_recall:.1%}"},
])
quality

**Observation:** Neither function is a trained ML classifier — they're rule-based heuristics. But that's the point: the operational graph pattern separates *where the decision is made* (the Function) from *how the graph is structured* (the schema) and *who acts on it* (the Action + governance). You can swap in a trained model for the function layer without touching anything else.

---

## 4. Decision Latency

How quickly can the system answer "what is the risk for entity X?"

In [ ]:
# Single-entity latency: time to fetch from Neo4j + compute risk
N_TRIALS = 20

ai4i_times = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    rows = graph.run("""
        MATCH (r:ToolRun {run_id: 'RUN_05000'})
        RETURN r.air_temp_k AS air_temp_k, r.process_temp_k AS process_temp_k,
               r.rotational_speed_rpm AS rotational_speed_rpm, r.torque_nm AS torque_nm,
               r.tool_wear_min AS tool_wear_min, r.machine_id AS machine_id
    """)
    ai4i_fn.compute(rows[0])
    ai4i_times.append((time.perf_counter() - t0) * 1000)

secom_times = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    rows = graph.run("""
        MATCH (lot:Lot {lot_id: 'LOT_0500'})
        OPTIONAL MATCH (lot)-[r:TRIGGERED_ALARM]->(:SPCAlarm)
        RETURN lot.n_spc_alarms AS n_spc_alarms,
               lot.sensor_na_rate AS sensor_na_rate,
               coalesce(max(abs(r.sigma_deviation)), 0.0) AS max_sigma_dev
    """)
    secom_fn.compute(rows[0])
    secom_times.append((time.perf_counter() - t0) * 1000)

latency = pd.DataFrame([
    {"Demo": "AI4I (ToolRun)",   "Median ms": f"{pd.Series(ai4i_times).median():.1f}",
     "P95 ms": f"{pd.Series(ai4i_times).quantile(0.95):.1f}",
     "Query type": "Simple node lookup + 5-rule compute"},
    {"Demo": "SECOM (Lot)",      "Median ms": f"{pd.Series(secom_times).median():.1f}",
     "P95 ms": f"{pd.Series(secom_times).quantile(0.95):.1f}",
     "Query type": "Node + max(rel.property) aggregate + 3-rule compute"},
])
print(latency.to_string(index=False))

In [ ]:
# Batch scoring latency
t0 = time.perf_counter()
all_runs = graph.run("""
    MATCH (r:ToolRun)
    RETURN r.air_temp_k AS air_temp_k, r.process_temp_k AS process_temp_k,
           r.rotational_speed_rpm AS rotational_speed_rpm, r.torque_nm AS torque_nm,
           r.tool_wear_min AS tool_wear_min, r.machine_id AS machine_id
""")
_ = [ai4i_fn.compute(r) for r in all_runs]
ai4i_batch_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
all_lots = graph.run("""
    MATCH (lot:Lot)
    OPTIONAL MATCH (lot)-[r:TRIGGERED_ALARM]->(:SPCAlarm)
    WITH lot, max(abs(r.sigma_deviation)) AS max_sigma
    RETURN lot.n_spc_alarms AS n_spc_alarms,
           lot.sensor_na_rate AS sensor_na_rate,
           coalesce(max_sigma, 0.0) AS max_sigma_dev
""")
_ = [secom_fn.compute(r) for r in all_lots]
secom_batch_ms = (time.perf_counter() - t0) * 1000

batch_latency = pd.DataFrame([
    {"Demo": "AI4I",  "Entities": 10_000, "Batch time (ms)": f"{ai4i_batch_ms:.0f}",
     "Per-entity (ms)": f"{ai4i_batch_ms/10000:.2f}"},
    {"Demo": "SECOM", "Entities": 1_567,  "Batch time (ms)": f"{secom_batch_ms:.0f}",
     "Per-entity (ms)": f"{secom_batch_ms/1567:.2f}"},
])
print(batch_latency.to_string(index=False))

**Observation:** Rule-based functions are extremely fast — sub-millisecond per entity once data is in-memory. The bottleneck is the Neo4j round-trip, not the computation. SECOM's query is slower per entity because it aggregates over relationship properties. 

In production, you'd store the scored `risk_score` and `risk_level` as derived properties (as done in the "Score All Runs/Lots" pages) so reads are O(1) lookups.

---

## 5. Action Safety (Traceability)

Every governed action writes an audit entry. Here's the full trace.

In [ ]:
with SessionLocal() as db:
    rows = db.execute(
        text("""
            SELECT action_name, actor, reason, outcome, error, created_at
            FROM audit_log
            ORDER BY id DESC
            LIMIT 30
        """)
    ).fetchall()

audit_df = pd.DataFrame([dict(r._mapping) for r in rows])
audit_df

In [ ]:
# Summary: outcomes across both actions
with SessionLocal() as db:
    rows = db.execute(
        text("""
            SELECT action_name, outcome, count(*) AS n
            FROM audit_log
            GROUP BY action_name, outcome
            ORDER BY action_name, outcome
        """)
    ).fetchall()

audit_summary = pd.DataFrame([dict(r._mapping) for r in rows])
print(audit_summary.to_string(index=False))

# Governance comparison table
governance = pd.DataFrame([
    {"Property": "Action name",       "AI4I": "trigger_maintenance",       "SECOM": "hold_lot"},
    {"Property": "Required role",     "AI4I": "engineer",                  "SECOM": "engineer"},
    {"Property": "Precondition 1",    "AI4I": "ToolRun must exist",        "SECOM": "Lot must exist"},
    {"Property": "Precondition 2",    "AI4I": "Not already scheduled",     "SECOM": "Not already on hold"},
    {"Property": "Graph write",       "AI4I": "maintenance_scheduled=true","SECOM": "lot_on_hold=true"},
    {"Property": "External writeback","AI4I": "mes_mock.jsonl",            "SECOM": "mes_mock.jsonl"},
    {"Property": "Audit logged",      "AI4I": "Yes (Postgres)",            "SECOM": "Yes (Postgres)"},
    {"Property": "Rollback defined",  "AI4I": "Yes (REMOVE properties)",  "SECOM": "Yes (REMOVE properties)"},
])
governance

**Observation:** The governance structure is identical across both demos because it lives in `core/actions/base.py`, not in the demo code. This is the point: the pattern is reusable. Adding a third demo would require writing only schema + function + action — not rebuilding the governance plumbing.

---

## 6. Operator Effort Comparison

How much work does it take an operator to go from "something is wrong" to "action taken"?

In [ ]:
effort = pd.DataFrame([
    {"Step": "Identify at-risk entity",
     "Without operational graph": "Pull report, scan spreadsheet, apply mental model",
     "With Track B": "Risk Dashboard shows sorted list, pre-scored"},
    {"Step": "Understand why it's risky",
     "Without operational graph": "Cross-reference logs, check multiple systems",
     "With Track B": "risk_factors dict shows exact rules triggered"},
    {"Step": "Verify preconditions",
     "Without operational graph": "Check CMMS/MES manually for existing work orders",
     "With Track B": "Precondition check is automatic (graph query)"},
    {"Step": "Take action",
     "Without operational graph": "Navigate to MES, create work order, enter details",
     "With Track B": "One button click in UI → API call → graph + MES updated"},
    {"Step": "Record decision",
     "Without operational graph": "Manually log in ticketing system (if at all)",
     "With Track B": "Automatic — every call writes actor/reason/outcome to audit log"},
    {"Step": "Review what was done",
     "Without operational graph": "Search tickets, emails, CMMS history",
     "With Track B": "Audit Log page shows full history with reasons"},
])
effort

---

## 7. When Is This Pattern Worth It?

The operational context graph adds real complexity. It's worth it when **all three** of these are true:

| Condition | AI4I | SECOM | Typical real factory? |
|---|---|---|---|
| Entities have meaningful relationships to each other | ✅ Machine → ToolRun → FailureMode | ✅ Lot → SPCAlarm features | Usually yes |
| Decisions require checking state across multiple entities | ✅ Is this machine type prone to this failure? | ✅ Which alarms co-occur in failing lots? | Usually yes |
| Actions must be governed and audited | ✅ | ✅ | Regulatory requirement in most fabs |

**When a simpler approach (SQL + REST + dashboard) is sufficient:**
- Single entity type with no meaningful cross-entity queries
- Actions don't need precondition checks against live state
- No traceability requirement
- Team is not comfortable with graph databases

In [ ]:
# Radar chart: operational graph vs simple approach on 5 dimensions
import plotly.graph_objects as go

categories = [
    "Traceability", "Cross-entity\nquerying", "Action safety",
    "Setup speed", "Team accessibility"
]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[5, 5, 5, 2, 3],  # operational graph
    theta=categories, fill='toself',
    name='Operational context graph',
    line_color='steelblue',
))
fig.add_trace(go.Scatterpolar(
    r=[2, 2, 2, 5, 5],  # simple SQL+dashboard
    theta=categories, fill='toself',
    name='SQL + REST + Dashboard',
    line_color='coral',
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
    title="Operational graph vs. simpler approach — where each wins",
    showlegend=True,
)
fig.show()

---

## 8. Key Observations

1. **The pattern scales across domains cleanly.** The same `core/` framework handled both a simple 5-feature machine dataset and a 590-feature semiconductor dataset. The only diff was schema + function + action.

2. **Track A is underrated.** Even before adding functions and actions, the graph structure of SECOM revealed that SPC alarms cluster on specific features. That's insight you get for free from the entity design.

3. **Functions are the honest layer.** Rule-based functions make their logic explicit and testable. The SECOM function has lower precision than AI4I because the dataset's failure signal is weaker — and that's visible in the numbers, not hidden in a model.

4. **Governance is boilerplate once.** The precondition → execute → audit → rollback pipeline was written once in `core/actions/base.py` and never touched again. Both demos got it for free.

5. **ID harmonization is the hardest part.** In both demos, the first real work was deciding what a `run_id` and `lot_id` look like, and making them consistent across CSVs, Neo4j nodes, and audit records. In a real factory, this is a months-long problem (tool IDs, lot IDs, chamber IDs, recipe IDs all come from different systems).

In [ ]:
graph.close()
print("Done.")